# 3. Generating RAG Answers (A -> Q -> A')

Retrieval metrics only test **search**. Next we test the **full RAG pipeline**:
search -> prompt -> LLM answer.

The **A -> Q -> A'** recipe:
- **A** = original FAQ answer (ground truth)
- **Q** = the synthetic question (from notebook 1)
- **A'** = the answer our RAG pipeline produces for Q

We save `(Q, A', A)` triples. The *next* notebook judges whether A' matches A.

In [1]:
import pandas as pd
df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

from ingest import load_faq_data, build_index
documents = [d for d in load_faq_data() if d["course"] == "llm-zoomcamp"]
index = build_index(documents)
doc_idx = {d["id"]: d for d in documents}  # quick id -> doc lookup

In [2]:
from dotenv import load_dotenv
import anthropic
load_dotenv()
client = anthropic.Anthropic()

In [3]:
# RAGWithUsage = RAGBase that records token usage per call (so we can price runs).
from evaluation_utils import RAGWithUsage
assistant = RAGWithUsage(index=index, llm_client=client, course="llm-zoomcamp")

In [4]:
# Answer ONE question end-to-end through the RAG pipeline.
q = ground_truth[10]
answer = assistant.rag(q["question"])
print("cost so far: $%.6f" % assistant.total_cost())
print(answer)

cost so far: $0.000884
Based on the provided context, you can find the link to watch the live workshop sessions in the following places:

*   **Telegram and Slack:** The video URL is posted in the announcements channel before the session begins.
*   **YouTube:** You can also watch the sessions live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).


In [5]:
# The ground-truth answer A (from the doc the question came from):
answer_orig = doc_idx[q['document']]['answer']
answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [6]:
# Build the (Q, A', A) triple.
rag_result = {
    "question": q["question"],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": q["document"],
}
rag_result

{'question': 'Where can I find the link to watch the live workshop sessions?',
 'answer_llm': 'Based on the provided context, you can find the link to watch the live workshop sessions in the following places:\n\n*   **Telegram and Slack:** The video URL is posted in the announcements channel before the session begins.\n*   **YouTube:** You can also watch the sessions live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

## Run over many questions in parallel

> **Cost control:** set `N_RAG`. The course ran all 395 (~$0.34 with gpt-5.4-mini).
> Here we generate a live batch with glm-5.2; bump to `len(ground_truth)` for the
> full run.

In [7]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

N_RAG = 50  # bump to len(ground_truth) for the full run
assistant.reset_usage()

def generate_rag_answer(rec):
    original_doc = doc_idx[rec["document"]]
    return {
        "question": rec["question"],
        "answer_llm": assistant.rag(rec["question"]),
        "answer_orig": original_doc["answer"],
        "document": rec["document"],
    }

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth[:N_RAG], generate_rag_answer)

print("cost: $%.6f" % assistant.total_cost())

  0%|          | 0/50 [00:00<?, ?it/s]

cost: $0.041656


In [8]:
df_results = pd.DataFrame(results)
df_results.head()
pd.DataFrame(results).to_csv("data/rag-answers.csv", index=False)
print("saved", len(results), "rows to data/rag-answers.csv")

saved 50 rows to data/rag-answers.csv


We now have `(Q, A', A)` triples. Time to judge whether A' is correct.